In [1]:
!pip install torch torchvision torchaudio
!pip install pandas numpy matplotlib tqdm
!pip install opencv-python
!pip install scikit-learn
!pip install ultralytics

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [1]:
import os
import cv2
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torchvision import transforms
from torchvision import models

from sklearn.model_selection import train_test_split

from tqdm import tqdm
import datetime

In [2]:
class UTKDataset(Dataset):

    def __init__(self, image_paths, transform=None):

        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        path = self.image_paths[idx]

        filename = os.path.basename(path)

        age = int(filename.split('_')[0])

        gender = int(filename.split('_')[1])

        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image,\
               torch.tensor(age,dtype=torch.float32),\
               torch.tensor(gender,dtype=torch.long)

In [3]:
train_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.RandomRotation(10),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

val_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [4]:
dataset_path = r"D:\ML-PROJECTS\Senior-Citizen-analysis\dataset\age\UTKFace"

images = []

for file in os.listdir(dataset_path):

    if file.endswith(".jpg"):

        images.append(
            os.path.join(dataset_path,file)
        )

print("Total Images:",len(images))

Total Images: 23708


In [5]:
train_imgs, val_imgs = train_test_split(
    images,
    test_size=0.2,
    random_state=42
)

train_dataset = UTKDataset(
    train_imgs,
    train_transform
)

val_dataset = UTKDataset(
    val_imgs,
    val_transform
)

In [6]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

In [7]:
class AgeGenderModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.backbone = models.efficientnet_b0(
            weights="IMAGENET1K_V1"
        )

        in_features = self.backbone.classifier[1].in_features

        self.backbone.classifier = nn.Identity()

        self.age_head = nn.Sequential(
            nn.Linear(in_features,256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256,1)
        )

        self.gender_head = nn.Sequential(
            nn.Linear(in_features,128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128,2)
        )

    def forward(self,x):

        feat = self.backbone(x)

        age = self.age_head(feat)

        gender = self.gender_head(feat)

        return age,gender

In [39]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = AgeGenderModel().to(device)

print(device)

cuda


In [25]:
age_loss_fn = nn.MSELoss()

gender_loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [12]:
best_acc = 0

patience = 8          # Stop if no improvement for 5 epochs
counter = 0

for epoch in range(50):   # Increased from 25 to 50

    model.train()

    train_loss = 0

    for imgs, ages, genders in tqdm(train_loader):

        imgs = imgs.to(device)
        ages = ages.to(device)
        genders = genders.to(device)

        pred_age, pred_gender = model(imgs)

        loss_age = age_loss_fn(
            pred_age.squeeze(),
            ages
        )

        loss_gender = gender_loss_fn(
            pred_gender,
            genders
        )

        loss = loss_age + loss_gender

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/50] | "
        f"Train Loss: {avg_train_loss:.4f}"
    )

    # Validation
    correct = 0
    total = 0

    model.eval()

    with torch.no_grad():

        for imgs, ages, genders in val_loader:

            imgs = imgs.to(device)
            genders = genders.to(device)

            _, pred_gender = model(imgs)

            pred = pred_gender.argmax(dim=1)

            correct += (pred == genders).sum().item()

            total += genders.size(0)

    acc = correct / total

    print(f"Validation Accuracy: {acc:.4f}")

    # Save Best Model
    if acc > best_acc:

        best_acc = acc
        counter = 0

        torch.save(
            model.state_dict(),
            "models/best_model.pth"
        )

        print(f"✅ Best Model Saved (Acc: {best_acc:.4f})")

    else:

        counter += 1

        print(
            f"No improvement for {counter}/{patience} epochs"
        )

        if counter >= patience:

            print(
                f"\n🛑 Early Stopping Triggered "
                f"at Epoch {epoch+1}"
            )
            break

print(f"\nBest Validation Accuracy: {best_acc:.4f}")

100%|██████████| 297/297 [06:59<00:00,  1.41s/it]


Epoch [1/50] | Train Loss: 379.1496
Validation Accuracy: 0.7472
✅ Best Model Saved (Acc: 0.7472)


100%|██████████| 297/297 [03:28<00:00,  1.42it/s]


Epoch [2/50] | Train Loss: 92.0763
Validation Accuracy: 0.7857
✅ Best Model Saved (Acc: 0.7857)


100%|██████████| 297/297 [03:17<00:00,  1.50it/s]


...

Epoch [48/50] | Train Loss: 14.8993
Validation Accuracy: 0.9019
No improvement for 3/8 epochs


100%|██████████| 297/297 [02:44<00:00,  1.80it/s]


Epoch [49/50] | Train Loss: 14.8666
Validation Accuracy: 0.8992
No improvement for 4/8 epochs


100%|██████████| 297/297 [02:44<00:00,  1.81it/s]


Epoch [50/50] | Train Loss: 14.8798
Validation Accuracy: 0.9053
✅ Best Model Saved (Acc: 0.9053)

Best Validation Accuracy: 0.9053


In [26]:
from ultralytics import YOLO

face_model = YOLO(r"D:\ML-PROJECTS\Senior-Citizen-analysis\models\face.pt")

In [27]:
model = AgeGenderModel()

model.load_state_dict(
    torch.load(
        "models/best_model.pth",
        map_location=device
    )
)

model.to(device)

model.eval()

AgeGenderModel(
  (backbone): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU(inplace=True)
              (sca

In [28]:
csv_file = "visits.csv"

if not os.path.exists(csv_file):

    pd.DataFrame(
        columns=[
            "Date",
            "Time",
            "Age",
            "Gender",
            "Status"
        ]
    ).to_csv(
        csv_file,
        index=False
    )

In [ ]:
import cv2
import torch
import pandas as pd
import datetime
from PIL import Image

# Open webcam
cap = cv2.VideoCapture(0)

# Check webcam
if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

logged = set()

while True:

    # Read frame
    ret, frame = cap.read()

    if not ret or frame is None:
        print("Failed to capture frame.")
        continue

    # Face detection
    results = face_model.predict(
        source=frame,
        verbose=False
    )

    h, w = frame.shape[:2]

    for r in results:

        boxes = r.boxes.xyxy.cpu().numpy()

        for box in boxes:

            x1, y1, x2, y2 = map(int, box)

            # Keep coordinates inside image
            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(w, x2)
            y2 = min(h, y2)

            # Skip invalid boxes
            if x2 <= x1 or y2 <= y1:
                continue

            # Crop face
            face = frame[y1:y2, x1:x2]

            if face.size == 0:
                continue

            try:
                # BGR -> RGB
                img = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

                img = Image.fromarray(img)

                img = val_transform(img)

                img = img.unsqueeze(0).to(device)

                # Prediction
                with torch.no_grad():
                    age, gender = model(img)

                age = int(age.item())
                gender = gender.argmax(1).item()

            except Exception as e:
                print("Prediction Error:", e)
                continue

            gender_text = "Male" if gender == 0 else "Female"

            status = "Senior Citizen" if age >= 60 else "Normal"

            label = f"{gender_text} | {age} | {status}"

            # Draw rectangle
            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            # Draw text
            cv2.putText(
                frame,
                label,
                (x1, max(30, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

            # Save senior citizens only
            if age >= 60:

                now = datetime.datetime.now()

                key = (
                    gender_text,
                    now.strftime("%Y-%m-%d"),
                    now.strftime("%H:%M")
                )

                if key not in logged:

                    df = pd.DataFrame([{
                        "Date": now.strftime("%Y-%m-%d"),
                        "Time": now.strftime("%H:%M:%S"),
                        "Age": age,
                        "Gender": gender_text,
                        "Status": status
                    }])

                    df.to_csv(
                        csv_file,
                        mode="a",
                        header=False,
                        index=False
                    )

                    logged.add(key)

    # Show frame
    cv2.imshow("Senior Citizen Detection", frame)

    # ESC to exit
    key = cv2.waitKey(1) & 0xFF

    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()